# Reproducing Jones et al. (2019) on CRCNS hc-26

**Paper.** Jones, Gillespie, Yoon, Frank and Huang (2019), "Early Hippocampal Sharp-Wave Ripple Deficits Predict Later Learning and Memory Impairments in an Alzheimer's Disease Mouse Model", Cell Reports 29, 2123-2133, doi:10.1016/j.celrep.2019.10.056.

**Claims file.** `claims/jones-2019.yaml`, written from the version of record with its supplemental information (Tables S1 to S4) and checked by a second reader.

**Data set and version.** CRCNS hc-26 (doi:10.6080/K0D798NB), read anonymously from the public bucket `crcnsarchive` under `hc-26/live/data/`. CRCNS data sets are not versioned, so the version is recorded as `unversioned`. The paper cites the data set by DOI only, and the bucket holds a single published copy (`live`; `orig/v0_60` holds only an older data descriptor), so that copy is the one used. The MD5 sums from the data set's `checksums.md5` for the files read are: `Cohort_1/E3KI.tar.gz` 190a898a787a89d4f9b6695b72d15ab0, `Cohort_1/E4KI.tar.gz` cf00e4427b44722fdebe93256c3f9afb, `Cohort_2/5-8mo/E3KI.tar.gz` b2144eade67d375f97a3485e6fc9d62c, `Cohort_2/5-8mo/E4KI.tar.gz` 91c27b0d9647a8a7d7e6a5f7c7b52529, `Cohort_2/9-11mo/E3KI.tar.gz` 6cac17fb2e6d8bad6b27e36d2e8442cc, `Cohort_2/9-11mo/E4KI.tar.gz` e742bc81377de83ec1db268a5731b7db, `Cohort_2/13-17mo/E3KI.tar.gz` df69e0160f3e174d3d7f945ee3cc99c2, `Cohort_2/13-17mo/E4KI.tar.gz` 75bf85e4291eefddec295fba78a8c67d, `Cohort_1/Behavior.tar.gz` 94b86383d1fad57be52b13f7aed96278, `Cohort_2/Behavior.tar.gz` 1613ac57f60b16244d4a513827b46723, `Cohort_1/animaldef.m` 8d55eb69ff7ce5fc9713ae0eabf8cd01, `Cohort_2/animaldef.m` 7d7906ed13af377d9d9c0654eae4d917. The Prism file `All_Figures.pzfx` is not read (it is Prism's binary format despite the XML extension).

**Outside input.** The ripple filter kernels of the paper's code (`Preprocessing/agripplefilter.mat`, `agripplefilter3.mat`), read from https://github.com/emilyasterjones/Jones_2019 at commit ca056a835631a8fea44a30cbf9845b53457dcbd1 (the repository the paper links as SWR-predictions). They are a method detail of the paper, not data.

**How the data are read.** The LFP is in nine tar.gz files (6 to 22 GB each, 108 GB in all). Each is streamed once, start to end, and reduced session by session (`swr.py`); four files are read in parallel. This takes about 40 minutes and is the bulk of the run time. Behavior comes from the two `Behavior.tar.gz` files (`behavior.py`). The statistics are in `stats.py` and the per-mouse tables in `analysis.py`.

The table of claims and outcomes is in the summary at the end. Blocked claims are listed in the cell below with their reason.


In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
import time, resource, warnings
import numpy as np, pandas as pd
import swr, behavior, stats, analysis as A
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 200)
T0 = time.time()
OUT = {}
CL = {}
def record(cid, outcome, measured=None, assumed=False, **kw):
    OUT[cid] = dict(outcome=outcome, measured=measured, assumed=assumed, **kw)
    print(f"[{cid}] outcome: {outcome}" + (" (rests on an assumption)" if assumed else "") + f"; measured: {measured}")
BLOCKED = {
    "table-s4-young-no-prediction": ("metadata_insufficient", "the 6-9 month MWM file identifies mice by its own MWM ID, which no file in the data set maps to the recorded animals"),
}
for k, v in BLOCKED.items():
    record(k, "blocked", blocked_by=v[0], why=v[1])


## Streaming and reducing the LFP

All nine LFP files are streamed and reduced here (see `swr.py` for every parameter and its source).

In [ ]:
t = time.time()
RES = swr.run_all(workers=4)
for r in RES:
    print(r["key"], "animals:", len(r["animals"]), "seconds:", round(r["seconds"]), "stream reopened:", r["reopened"])
print("extraction minutes:", round((time.time() - t) / 60, 1), "peak memory of this process (MB):", resource.getrusage(resource.RUSAGE_SELF).ru_maxrss // 1000)
SESS = A.session_table(swr.session_rows(RES))
MICE = A.mouse_table(SESS)


## Data check

In [ ]:
S = SESS
print("Sessions and mice per cohort, age group and genotype (all deposited, and used):")
print(S.groupby(["cohort", "age_group", "genotype"]).agg(mice=("key", "nunique"), sessions=("session", "size"),
      fail=("fail", "sum"), short_immobility=("short_immobility", "sum"), used=("used", "sum")))
print("\nMice with at least one used session:")
print(MICE.groupby(["cohort", "age_group", "genotype"]).size().unstack())
print("\nTask 'descript' notes (all sessions):"); print(S.task_descript.value_counts().head(40))
print("\nTask env values:", S.task_env.value_counts().to_dict())
print("\nPer-session ranges (used sessions):")
U = S[S.used]
print(U[["duration_s", "immobile_s", "base", "sd", "n_candidates", "n_swr_5sd", "n_noise", "rate_5sd", "vel_max", "age_months"]].describe().T.round(3))
print("\nAge by group (months, task.age):"); print(U.groupby(["cohort", "age_group", "genotype"]).age_months.agg(["min", "max"]))
print("\nSWRs counted by the Methods' wording (direct detection at 5 SD) and by the code's 3 SD detection with maxthresh >= 5, used sessions: identical in",
      int((U.n_swr_direct5 == U.n_swr_5sd).sum()), "of", len(U), "sessions")
labs = pd.Series([f"{a}|{l}" for r in RES for an in r["animals"].values() for s in an["sessions"].values() for (_, a, l) in s.get("groups", {}).get("labels", [])])
print("\nchinfo area|layer labels (site-sessions):"); print(labs.value_counts().to_string())
print("\nDetection site found in every session:", int(S.get("no_detection_site", pd.Series(False, index=S.index)).fillna(False).sum()) == 0,
      "; sessions without position:", int(S.no_position.sum()))
mu = S.mua_unit.dropna()
print("\nMUA spike time unit (the unit that puts every session's spikes inside its recording range):", mu.value_counts().to_dict(),
      "; sessions with MUA:", len(mu), "of", int((S.cohort == 1).sum()), "Cohort 1 sessions")


Clock check. The LFP, the position and the MUA are in seconds on one clock according to the descriptor (eeg.starttime, pos.time, task.start in ms). The check below compares the start of each stream, and then uses a contrast: SWRs occur during immobility, so the fraction of candidate 5 SD events that fall in immobile time should exceed the fraction of time that is immobile, and the excess should fall when the immobility intervals are shifted by a minute or more.

In [ ]:
rows = []
for r in RES:
    for an, a in r["animals"].items():
        for s, ses in a["sessions"].items():
            if "pos_range" not in ses: continue
            ev = ses["det_events"]; sel = ev["maxthresh"] >= 5
            mid = ev["peak"][sel]; iv = ses["immobile"]; n = ses["n"]
            tfrac = (iv[:, 1] - iv[:, 0]).sum() / n if len(iv) else np.nan
            out = {"animal": an, "session": s, "pos_start_minus_lfp_start_s": ses["pos_range"][0] - ses["t0"],
                   "pos_end_minus_lfp_end_s": ses["pos_range"][1] - (ses["t0"] + n / 1000), "time_immobile": tfrac}
            for shift in (0, 60, 300):
                sh = np.clip(iv + shift * 1000, 0, n)
                out[f"events_immobile_shift{shift}s"] = swr._in_intervals(mid, sh).mean() if len(mid) else np.nan
            rows.append(out)
CK = pd.DataFrame(rows)
print(CK.describe().T.round(3))
for shift in (0, 60, 300):
    print(f"shift {shift:3d} s: median (fraction of events in immobile time - fraction of time immobile) =",
          round(float((CK[f'events_immobile_shift{shift}s'] - CK.time_immobile).median()), 3))


In [ ]:
print("Behavior: mice per table and genotype")
TAB = {"Cohort 1 MWM": A.mwm_metrics(1), "Cohort 2 MWM 6-9 mo": A.mwm_metrics(2, "6-9mo"),
       "Cohort 2 MWM 14-18 mo": A.mwm_metrics(2, "14-18mo"), "APA": A.apa_metrics()}
for k, v in TAB.items():
    print(f"{k:24s}", v.groupby("genotype").size().to_dict(), " linked to a recorded animal:", int(v.key.notna().sum()))
for k, f in {"open field": behavior.open_field, "elevated plus": behavior.elevated_plus, "hot plate": behavior.hot_plate,
             "visible MWM 14-18": lambda: behavior.mwm_visible(2, "14-18mo")}.items():
    print(f"{k:24s}", f().groupby("genotype").size().to_dict())
LT = behavior.link_table()
print("Genotype disagreements between sources (link table):", int(LT.genotype_disagree.sum()))
m1 = TAB["Cohort 1 MWM"]
print("Ranges of MWM metrics (Cohort 1):"); print(m1[A.MWM22].describe().T[["count", "min", "max"]].round(2))
a = TAB["APA"]; print("Ranges of APA metrics:"); print(a.drop(columns=["mouse", "name", "genotype", "key"]).describe().T[["count", "min", "max"]].round(2))


In [ ]:
# Link check for Cohort 2: the 14-18 month MWM file's 'Sheet1' lists per-trial values under mouse names;
# they should track the escape latencies of the blind code that the manifest gives each name.
book = behavior._excel(2, "mwm_14-18mo"); s1 = behavior._sheet(book, "sheet1")
h = behavior.mwm_hidden(2, "14-18mo"); h["tidx"] = (h.day - 1) * 4 + h.trial
lat = h.pivot_table(index="tidx", columns="name", values="latency_s")
for j, nm in enumerate(s1.iloc[0, 1:].tolist()):
    if not isinstance(nm, str): continue
    v = pd.to_numeric(s1.iloc[1:29, j + 1], errors="coerce").values
    rs = {c: np.corrcoef(*[a[np.isfinite(v) & np.isfinite(l)] for a in (v, l)])[0, 1] for c in lat.columns for l in [lat[c].reindex(range(1, 29)).values]}
    own = [r_ for c, r_ in rs.items() if behavior.norm_name(c) == behavior.norm_name(nm)]
    best = max(rs, key=rs.get)
    print(f"{nm:16s} own mouse r = {own[0] if own else float('nan'):.2f}; best match {best} (r = {rs[best]:.2f})")


Per-mouse tables used below. `rate_5sd` is SWR abundance (Hz) by method (A); `sg_<region>_<trigger>_<band>` is SG power during SWRs (z) averaged per session then over sessions (the Methods' reading of method (B)), and `sgpool_...` the code's reading, SWRs pooled over sessions per site. Trigger `5sd` is the SWRs of method (A), `3sd` the code's minstd 3; band `30-50` holds 31.3, 39.1 and 46.9 Hz, `bins4-7` adds 54.7 Hz.

In [ ]:
def ephys(cohort, age):
    return MICE[(MICE.cohort == cohort) & (MICE.age_group == age)].set_index("key")
C1 = ephys(1, "12-18mo"); R13 = ephys(2, "13-17mo"); R9 = ephys(2, "9-11mo"); R5 = ephys(2, "5-8mo")
MWM1 = TAB["Cohort 1 MWM"].dropna(subset=["key"]).set_index("key")
MWM2 = TAB["Cohort 2 MWM 14-18 mo"].dropna(subset=["key"]).set_index("key")
APA_W = TAB["APA"].dropna(subset=["key"]).set_index("key")
APA_ALL = A.apa_metrics("dist_per_bout_cm").dropna(subset=["key"]).set_index("key")
SG = "sg_ca3_5sd_30-50"
SG_READINGS = {"per session, 5 SD, 30-50 Hz (Methods)": "sg_ca3_5sd_30-50", "pooled, 5 SD, 30-50 Hz (code pooling)": "sgpool_ca3_5sd_30-50"}
SG_VARIANTS = {"pooled, 3 SD, bins 4-7 (code)": "sgpool_ca3_3sd_bins4-7", "per session, 3 SD, 30-50 Hz": "sg_ca3_3sd_30-50", "per session, 5 SD, bins 4-7": "sg_ca3_5sd_bins4-7"}
def join(e, b, cols_e, geno="E4"):
    d = e[e.genotype == geno][cols_e].join(b.drop(columns=["genotype"], errors="ignore"), how="inner")
    return d
print("Cohort 1 E4 with MWM:", len(join(C1, MWM1, ["rate_5sd"])), "; with a CA3 value:", int(join(C1, MWM1, [SG])[SG].notna().sum()))
print("Cohort 2 E4 13-17 mo with MWM 14-18:", len(join(R13, MWM2, ["rate_5sd"])), "; with CA3:", int(join(R13, MWM2, [SG])[SG].notna().sum()))
print("Cohort 2 E4 13-17 mo with APA:", len(join(R13, APA_W, ["rate_5sd"])), "; with CA3:", int(join(R13, APA_W, [SG])[SG].notna().sum()))
print("Cohort 2 E4 5-8 mo with MWM 14-18:", len(join(R5, MWM2, ["rate_5sd"])), "; with CA3:", int(join(R5, MWM2, [SG])[SG].notna().sum()),
      "; with APA:", len(join(R5, APA_W, ["rate_5sd"])), "; with CA3:", int(join(R5, APA_W, [SG])[SG].notna().sum()))
print("\nCohort 1 per mouse:"); print(C1[["genotype", "n_sessions", "age_months", "rate_5sd", "sg_ca3_5sd_30-50", "sgpool_ca3_5sd_30-50", "sg_ca1sr_5sd_30-50", "sg_dg_5sd_30-50", "base", "sd"]].round(3))


## fig1e-swr-abundance-genotype

> In the aged screen cohort, apoE4-KI mice have lower SWR abundance than apoE3-KI mice.

In [ ]:
e3, e4 = C1[C1.genotype == "E3"].rate_5sd, C1[C1.genotype == "E4"].rate_5sd
r = stats.ttest(e3, e4); print("main (method A):", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in r.items()})
for v in ["rate_3sd", "rate_direct5", "rate_125_5sd"]:
    rv = stats.ttest(C1[C1.genotype == "E3"][v], C1[C1.genotype == "E4"][v]); print(f"variant {v}: t({rv['df']}) = {rv['t']:.2f}, p = {rv['p']:.2g}, means {rv['mean_a']:.3f} vs {rv['mean_b']:.3f}")
print("E3 range", e3.min().round(3), e3.max().round(3), "; E4 range", e4.min().round(3), e4.max().round(3))
out = "reproduced" if (r["mean_b"] < r["mean_a"] and r["p"] < 1e-4) else ("partially_reproduced" if r["mean_b"] < r["mean_a"] and r["p"] < 0.05 else "not_reproduced")
record("fig1e-swr-abundance-genotype", out, measured=f"t({r['df']}) = {r['t']:.2f}, p = {r['p']:.1e}; E3 {r['mean_a']:.3f} Hz, E4 {r['mean_b']:.3f} Hz")


**Result:** Reproduced. E4 mice had lower SWR abundance than E3 mice, 0.113 against 0.245 Hz, t(27) = 8.21, p = 8.2e-09 (paper t(27) = 8.42, p < 0.0001). The 3 SD and 125-250 Hz variants agree in direction and significance.

## fig1f-sg-power-genotype

> In the aged screen cohort, SWR-associated SG power is lower in apoE4-KI than in apoE3-KI mice in CA1 stratum radiatum, CA3 and the dentate gyrus.

In [ ]:
res = {}
for rd in ["sg_{}_5sd_30-50", "sgpool_{}_5sd_30-50", "sgpool_{}_3sd_bins4-7", "sg_{}_3sd_30-50"]:
    line = []; ok = 0
    for reg, test in (("ca1sr", "t"), ("ca3", "t"), ("dg", "mw"), ("dgval", "mw")):
        c = rd.format(reg); a, b = C1[C1.genotype == "E3"][c].dropna(), C1[C1.genotype == "E4"][c].dropna()
        r = stats.ttest(a, b) if test == "t" else stats.mannwhitney(a, b)
        s = f"t({r['df']})={r['t']:.2f}" if test == "t" else f"U={r['U']:.0f}"
        low = b.mean() < a.mean() if test == "t" else b.median() < a.median()
        if reg != "dgval": ok += int(low and r["p"] < 0.05)
        line.append(f"{reg}: n {len(a)}/{len(b)}, {s}, p={r['p']:.2g}, E3 {a.mean():.2f} E4 {b.mean():.2f}")
    res[rd] = ok
    print(rd.replace("{}", "<region>"), "| regions with E4 lower, p<0.05 (ca1sr, ca3, dg):", ok); print("   ", "\n    ".join(line))
best = max(res.values())
out = "reproduced" if best == 3 else ("partially_reproduced" if best == 2 else "not_reproduced")
record("fig1f-sg-power-genotype", out, measured=f"{res['sg_{}_5sd_30-50']} of 3 regions under the Methods reading; best over readings {best} of 3")


**Result:** Reproduced. SG power during SWRs was lower in E4 mice in all three regions under every reading: CA1-sr t(27) = 6.48, CA3 t(27) = 4.82, DG U = 40 (p = 0.0054) under the Methods reading. The data's site labels give CA3 sites in all 29 mice, where the paper has 11 E3 and 13 E4 (t(22)), so the CA3 test has 5 more mice than the paper's.

## figs1a-mua-during-swrs

> Multi-unit activity in CA1 stratum pyramidale rises from before to during SWRs in both genotypes, with no genotype difference before or during SWRs.

In [ ]:
ok_all = {}
for pre in ["mua_pre400", "mua_pre400_300"]:
    ok = True; print(f"baseline reading {pre}:")
    for g in ("E3", "E4"):
        d = C1[C1.genotype == g]; r = stats.paired_t(d.mua_during, d[pre])
        print(f"  {g}: during {d.mua_during.mean():.1f} Hz vs before {d[pre].mean():.1f} Hz, paired t({r['df']}) = {r['t']:.2f}, p = {r['p']:.2g}")
        ok &= (r["p"] < 0.05) and r["mean_diff"] > 0
    for c in (pre, "mua_during"):
        r = stats.ttest(C1[C1.genotype == "E3"][c], C1[C1.genotype == "E4"][c])
        print(f"  genotype, {c}: t({r['df']}) = {r['t']:.2f}, p = {r['p']:.3f} (E3 {r['mean_a']:.1f}, E4 {r['mean_b']:.1f})"); ok &= r["p"] >= 0.05
    ok_all[pre] = ok
print("criterion holds per reading:", ok_all)
outs = set(ok_all.values())
out = "reproduced" if outs == {True} else ("not_reproduced" if outs == {False} else "reproduced")
record("figs1a-mua-during-swrs", out, assumed=len(outs) > 1, measured=str(ok_all))


**Result:** Reproduced. MUA rose from before to during SWRs in both genotypes (E3 paired t(12) = 5.30, E4 t(15) = 3.91, whole 400 ms baseline; the same with the -400 to -300 ms bin), and the genotypes did not differ before (p = 0.72) or during SWRs (p = 0.49).

## fig1g-mwm-impaired

> Aged apoE4-KI mice of the screen cohort learn the hidden platform location more slowly on the MWM than apoE3-KI mice, with higher escape latencies, significant on hidden days 1, 3 and 5.

In [ ]:
d = TAB["Cohort 1 MWM"]
long = d.melt(id_vars=["mouse", "genotype"], value_vars=[f"latency_d{k}" for k in range(1, 6)], var_name="day", value_name="lat").dropna()
long["day"] = long.day.str[-1].astype(int)
art = stats.art_anova_rm(long, "mouse", "genotype", "day", "lat")
for k, v in art.items(): print(k, {kk: v[kk] for kk in ("path", "n_subjects", "F", "df", "p")})
print("mean latency by genotype and day:"); print(long.groupby(["genotype", "day"]).lat.mean().unstack().round(1))
sig = {}
for k in range(1, 6):
    a, b = d[d.genotype == "E3"][f"latency_d{k}"], d[d.genotype == "E4"][f"latency_d{k}"]
    r = stats.mannwhitney(a, b); padj = stats.sidak(r["p"], 5); sig[k] = padj < 0.05
    print(f"day {k}: U = {r['U']:.0f}, p = {r['p']:.4f}, Sidak (5 days) p = {padj:.4f}, E3 median {a.median():.1f}, E4 median {b.median():.1f}")
g = art["between"]; slower = long[long.genotype == "E4"].lat.mean() > long[long.genotype == "E3"].lat.mean()
nsig = sum(sig[k] for k in (1, 3, 5))
out = "not_reproduced" if not (g["p"] < 0.05 and slower) else ("reproduced" if nsig == 3 else ("partially_reproduced" if nsig >= 1 else "partially_reproduced"))
record("fig1g-mwm-impaired", out, measured=f"genotype F = {g['F']:.2f}, p = {g['p']:.4f}; days significant after Sidak: {[k for k in sig if sig[k]]}")


**Result:** Reproduced. The aligned rank transform ANOVA gives a genotype effect F(1,37) = 13.08, p = 0.0009, as printed, with E4 slower; Sidak-adjusted Mann-Whitney tests are significant on days 1, 3 and 5 (adjusted p 0.022, 0.039, 0.030) and not on days 2 and 4.

## screen-88-comparisons

> In the screen cohort, 9 of 88 comparisons of 22 MWM metrics against 4 SWR properties are significant at alpha 0.05 in apoE4-KI mice, more than chance by a binomial test.

In [ ]:
props = {"abundance": "rate_5sd", "CA3": "sg_ca3_5sd_30-50", "CA1-sr": "sg_ca1sr_5sd_30-50", "DG": "sg_dg_5sd_30-50"}
def count88(props):
    tab = []
    for pn, pc in props.items():
        d = join(C1, MWM1, [pc])
        for m in A.MWM22:
            r = stats.pearson(d[pc], d[m]); tab.append((pn, m, r["n"], r["r"], r["p"]))
    return pd.DataFrame(tab, columns=["property", "metric", "n", "R", "p"])
T88 = count88(props); k = int((T88.p < 0.05).sum())
print(T88[T88.p < 0.05].round(3).to_string())
print(f"significant: {k} of {len(T88)}; one-sided binomial p = {stats.binom_greater(k, len(T88)):.4g}")
props_pool = {k_: v.replace("sg_", "sgpool_") for k_, v in props.items()}; props_pool["abundance"] = "rate_5sd"
T88p = count88(props_pool); kp = int((T88p.p < 0.05).sum()); print("pooled SG reading:", kp, "significant; binomial p =", round(stats.binom_greater(kp, 88), 4))
res = {"per session": k, "pooled": kp}
oks = {n_: (abs(v - 9) <= 1 and stats.binom_greater(v, 88) < 0.05) for n_, v in res.items()}
out = "reproduced" if all(oks.values()) else ("not_reproduced" if not any(oks.values()) else "reproduced")
record("screen-88-comparisons", out, assumed=len(set(oks.values())) > 1, measured=f"{k} of 88 (per-session SG), {kp} of 88 (pooled SG)")


**Result:** Not reproduced. 5 of the 88 correlations have p < 0.05 (the same 5 under the pooled SG reading), against 9 in the paper; the one-sided binomial p is 0.45, not below 0.05.

The correlation claims below follow the rule of `fig2b-swr-slope-days-1-2` (Pearson across the E4 mice with both measures; reproduced when R has the paper's sign with p < 0.05 and R is within tolerance; partially reproduced when sign and p hold but R is outside tolerance). For CA3 SG power the two readings that the claims file leaves open (per session or pooled over sessions, method B) are both run; when they give different outcomes, the Methods' per-session reading is adopted as an assumption. The code's other choices (3 SD trigger, 54.7 Hz bin) are shown as variants. Each line also gives the leave-one-out result (loocv.m) used by `robust-single-point`.

In [ ]:
LOO = {}
def corr_claim(cid, E, B, xcol, ycol, er, tol, geno="E4", extra=""):
    E, B = globals()[E] if isinstance(E, str) else E, globals()[B] if isinstance(B, str) else B
    readings = SG_READINGS if xcol == "SG" else {"method A": xcol}
    variants = SG_VARIANTS if xcol == "SG" else {"3 SD threshold (code variant)": "rate_3sd"}
    outs, first = {}, None
    for name, c in readings.items():
        d = join(E, B, [c], geno); r = A.corr_report(d[c], d[ycol], f"{name}: {c} vs {ycol}")
        outs[name] = A.judge_corr(r, er, tol); first = first or (r, name)
        LOO.setdefault(cid, r)
    for name, c in variants.items():
        d = join(E, B, [c], geno); A.corr_report(d[c], d[ycol], f"variant {name}: {c} vs {ycol}")
    vals = set(outs.values())
    out = outs[list(readings)[0]]
    r = first[0]
    record(cid, out, assumed=len(vals) > 1, measured=round(r["r"], 3), readings=outs, p=r["p"], n=r["n"])
    return r


## fig2b-swr-slope-days-1-2

> See claims file; rate_5sd against slope_1_2, expected R = -0.56 (tolerance 0.056).

In [ ]:
CL["fig2b-swr-slope-days-1-2"] = corr_claim("fig2b-swr-slope-days-1-2", "C1", "MWM1", "rate_5sd", "slope_1_2", -0.56, 0.056)

**Result:** Not reproduced. R = -0.471 has the paper's sign but p = 0.066 (n = 16; paper R = -0.56, p = 0.024). The 3 SD variant gives R = -0.332.

## figs1b-swr-slope-days-1-3

> See claims file; rate_5sd against slope_1_3, expected R = -0.66 (tolerance 0.066).

In [ ]:
CL["figs1b-swr-slope-days-1-3"] = corr_claim("figs1b-swr-slope-days-1-3", "C1", "MWM1", "rate_5sd", "slope_1_3", -0.66, 0.066)

**Result:** Partially reproduced. R = -0.572, p = 0.021 (n = 16): sign and significance hold, but R is outside the tolerance of -0.66 +- 0.066.

## fig2c-swr-latency-day-3

> See claims file; rate_5sd against latency_d3, expected R = -0.5 (tolerance 0.05).

In [ ]:
CL["fig2c-swr-latency-day-3"] = corr_claim("fig2c-swr-latency-day-3", "C1", "MWM1", "rate_5sd", "latency_d3", -0.5, 0.05)

**Result:** Not reproduced. R = -0.434, p = 0.093 (n = 16; paper R = -0.50, p = 0.046).

## fig2d-swr-overnight

> See claims file; rate_5sd against night_1, expected R = -0.64 (tolerance 0.064).

In [ ]:
CL["fig2d-swr-overnight"] = corr_claim("fig2d-swr-overnight", "C1", "MWM1", "rate_5sd", "night_1", -0.64, 0.064)

**Result:** Partially reproduced. R = -0.524, p = 0.037 (n = 16): sign and significance hold, R is outside -0.64 +- 0.064. The 3 SD variant gives R = -0.267, p = 0.32.

## fig2f-ca3-probe1-quadrant

> See claims file; SG against pct_quadrant_p1, expected R = 0.77 (tolerance 0.077).

In [ ]:
CL["fig2f-ca3-probe1-quadrant"] = corr_claim("fig2f-ca3-probe1-quadrant", "C1", "MWM1", "SG", "pct_quadrant_p1", 0.77, 0.077)

**Result:** Not reproduced. R = +0.487, p = 0.056 (per session) and +0.474, p = 0.063 (pooled), n = 16 (paper R = 0.77, n = 13). The data's CA3 labels include 3 E4 mice the paper excluded.

## figs1d-ca3-probe2-quadrant

> See claims file; SG against pct_quadrant_p2, expected R = 0.57 (tolerance 0.057).

In [ ]:
CL["figs1d-ca3-probe2-quadrant"] = corr_claim("figs1d-ca3-probe2-quadrant", "C1", "MWM1", "SG", "pct_quadrant_p2", 0.57, 0.057)

**Result:** Not reproduced. R = +0.356, p = 0.18 (n = 16; paper R = 0.57, p = 0.041).

## fig2g-ca3-probe1-crossings

> See claims file; SG against crossings_p1, expected R = 0.78 (tolerance 0.078).

In [ ]:
CL["fig2g-ca3-probe1-crossings"] = corr_claim("fig2g-ca3-probe1-crossings", "C1", "MWM1", "SG", "crossings_p1", 0.78, 0.078)

**Result:** Partially reproduced. R = +0.641, p = 0.0075 (n = 16) per session, +0.625 pooled: sign and significance hold, R is outside 0.78 +- 0.078.

## fig2h-ca3-probe1-auc

> See claims file; SG against auc_p1, expected R = -0.61 (tolerance 0.061).

In [ ]:
CL["fig2h-ca3-probe1-auc"] = corr_claim("fig2h-ca3-probe1-auc", "C1", "MWM1", "SG", "auc_p1", -0.61, 0.061)

**Result:** Not reproduced. R = -0.460, p = 0.073 (n = 16; paper R = -0.61, p = 0.027).

## figs1f-ca3-probe2-auc

> See claims file; SG against auc_p2, expected R = -0.59 (tolerance 0.059).

In [ ]:
CL["figs1f-ca3-probe2-auc"] = corr_claim("figs1f-ca3-probe2-auc", "C1", "MWM1", "SG", "auc_p2", -0.59, 0.059)

**Result:** Not reproduced. R = -0.270, p = 0.31 (n = 16; paper R = -0.59).

## rep-mwm-swr-slope-days-1-2

> See claims file; rate_5sd against slope_1_2, expected R = -0.675 (tolerance 0.068).

In [ ]:
CL["rep-mwm-swr-slope-days-1-2"] = corr_claim("rep-mwm-swr-slope-days-1-2", "R13", "MWM2", "rate_5sd", "slope_1_2", -0.675, 0.068)

**Result:** Not reproduced. In the 11 replication E4 mice, R = -0.021, p = 0.95 (paper R = -0.675).

## rep-mwm-swr-latency-day-3

> See claims file; rate_5sd against latency_d3, expected R = -0.614 (tolerance 0.061).

In [ ]:
CL["rep-mwm-swr-latency-day-3"] = corr_claim("rep-mwm-swr-latency-day-3", "R13", "MWM2", "rate_5sd", "latency_d3", -0.614, 0.061)

**Result:** Not reproduced. R = -0.113, p = 0.74 (n = 11; paper R = -0.614).

## rep-mwm-swr-overnight

> See claims file; rate_5sd against night_1, expected R = -0.652 (tolerance 0.065).

In [ ]:
CL["rep-mwm-swr-overnight"] = corr_claim("rep-mwm-swr-overnight", "R13", "MWM2", "rate_5sd", "night_1", -0.652, 0.065)

**Result:** Not reproduced. R = -0.087, p = 0.80 (n = 11; expected R = -0.652).

## rep-mwm-swr-slope-days-1-3

> See claims file; rate_5sd against slope_1_3, expected R = -0.773 (tolerance 0.077).

In [ ]:
CL["rep-mwm-swr-slope-days-1-3"] = corr_claim("rep-mwm-swr-slope-days-1-3", "R13", "MWM2", "rate_5sd", "slope_1_3", -0.773, 0.077)

**Result:** Not reproduced. R = -0.250, p = 0.46 (n = 11; expected R = -0.773).

## rep-mwm-ca3-probe1-quadrant

> See claims file; SG against pct_quadrant_p1, expected R = 0.896 (tolerance 0.09).

In [ ]:
CL["rep-mwm-ca3-probe1-quadrant"] = corr_claim("rep-mwm-ca3-probe1-quadrant", "R13", "MWM2", "SG", "pct_quadrant_p1", 0.896, 0.09)

**Result:** Not reproduced. R = +0.034, p = 0.94, in the 8 E4 mice whose deposited site labels include CA3 at 13-17 months (paper n = 10, R = 0.896).

## rep-mwm-ca3-probe1-crossings

> See claims file; SG against crossings_p1, expected R = 0.792 (tolerance 0.079).

In [ ]:
CL["rep-mwm-ca3-probe1-crossings"] = corr_claim("rep-mwm-ca3-probe1-crossings", "R13", "MWM2", "SG", "crossings_p1", 0.792, 0.079)

**Result:** Not reproduced. R = +0.199, p = 0.64 (n = 8; paper R = 0.792).

## rep-mwm-ca3-probe1-auc

> See claims file; SG against auc_p1, expected R = -0.698 (tolerance 0.07).

In [ ]:
CL["rep-mwm-ca3-probe1-auc"] = corr_claim("rep-mwm-ca3-probe1-auc", "R13", "MWM2", "SG", "auc_p1", -0.698, 0.07)

**Result:** Not reproduced. R = +0.415, p = 0.31 (n = 8), the opposite sign to the paper's -0.698.

## rep-mwm-ca3-probe2

> In the replication cohort, CA3 SG power during SWRs predicts percent time in the target quadrant and the area under the distance curve on probe 2.

In [ ]:
ra = corr_claim("rep-mwm-ca3-probe2-q", "R13", "MWM2", "SG", "pct_quadrant_p2", 0.655, 0.066)
rb = corr_claim("rep-mwm-ca3-probe2-a", "R13", "MWM2", "SG", "auc_p2", -0.713, 0.071)
oa, ob = OUT.pop("rep-mwm-ca3-probe2-q"), OUT.pop("rep-mwm-ca3-probe2-a")
held = [o["outcome"] == "reproduced" for o in (oa, ob)]
anyheld = [o["outcome"] in ("reproduced", "partially_reproduced") for o in (oa, ob)]
out = "reproduced" if all(held) else ("partially_reproduced" if any(anyheld) else "not_reproduced")
record("rep-mwm-ca3-probe2", out, assumed=oa["assumed"] or ob["assumed"], measured=[oa["measured"], ob["measured"]])


**Result:** Not reproduced. Percent time in quadrant on probe 2 R = +0.360, p = 0.38; area under the curve R = -0.285, p = 0.49 (n = 8).

## fig2e-learning-score

> In aged apoE4-KI mice of the screen cohort, z-scored SWR abundance accounts for 51% of the variance of the MWM learning performance score.

In [ ]:
def score_table(E, B, xcol, metrics, inv, pop, geno="E4"):
    d = E[[xcol, "genotype"]].join(B.drop(columns=["genotype"], errors="ignore"), how="inner").copy()
    d["x"] = d[xcol].where(d.genotype == geno)
    d["score"] = A.scores(d, metrics, inv, pop)
    return d[d.genotype == geno].dropna(subset=["x", "score"])
SC = {}
for pop in ("e4", "all"):
    d = score_table(C1, MWM1, "rate_5sd", A.MWM_LEARN, A.MWM_LEARN_INV, pop); SC[("learn", pop)] = d
    r = A.corr_report(stats.zscore(d.x), d.score, f"z-score population {pop}")
    SC[("learn_r", pop)] = r
LOO["fig2e-learning-score"] = SC[("learn_r", "e4")]
ok = [SC[("learn_r", p)]["r"] > 0 and SC[("learn_r", p)]["p"] < 0.05 and abs(SC[("learn_r", p)]["r2"] - 0.51) <= 0.051 for p in ("e4", "all")]
part = [SC[("learn_r", p)]["r"] > 0 and SC[("learn_r", p)]["p"] < 0.05 for p in ("e4", "all")]
out = "reproduced" if any(ok) else ("partially_reproduced" if any(part) else "not_reproduced")
record("fig2e-learning-score", out, measured=[round(SC[("learn_r", p)]["r2"], 3) for p in ("e4", "all")])


**Result:** Partially reproduced. The learning score correlates with z-scored SWR abundance, R = +0.612, p = 0.012 (n = 16), but R squared is 0.375 (0.377 with the all-mice z-score), outside 0.51 +- 0.051.

## fig2i-memory-precision-score

> In aged apoE4-KI mice of the screen cohort, z-scored CA3 SG power during SWRs accounts for 77% of the variance in the MWM memory precision performance score.

In [ ]:
res = {}
for sgc in SG_READINGS.values():
    for pop in ("e4", "all"):
        d = score_table(C1, MWM1, sgc, A.MWM_MEM, A.MWM_MEM_INV, pop); SC[("mem", pop, sgc)] = d
        r = A.corr_report(stats.zscore(d.x), d.score, f"{sgc}, z-score population {pop}"); res[(sgc, pop)] = r
ok = [r["r"] > 0 and r["p"] < 0.05 and abs(r["r"] - 0.844) <= 0.084 for r in res.values()]
part = [r["r"] > 0 and r["p"] < 0.05 for r in res.values()]
print("R squared compared with 0.71 (from the printed R and F) and 0.77 (text):", {k: round(v["r2"], 3) for k, v in res.items()})
r0 = res[(SG, "e4")]; LOO["fig2i-memory-precision-score"] = r0
out = "reproduced" if any(ok) else ("partially_reproduced" if any(part) else "not_reproduced")
record("fig2i-memory-precision-score", out, measured=[round(r0["r"], 3), round(r0["r2"], 3)])


**Result:** Partially reproduced. The memory precision score correlates with CA3 SG power, R = +0.641, p = 0.0074 (n = 16; R squared 0.41), positive and significant under all four readings, but R is outside 0.844 +- 0.084 under each (largest 0.660).

## screen-features-independent

> In aged apoE4-KI mice of the screen cohort, SWR abundance and SWR-associated CA3 SG power are not correlated, and measures of early learning speed and of probe memory are not correlated.

In [ ]:
d = C1[C1.genotype == "E4"]
r1 = A.corr_report(d.rate_5sd, d[SG], "abundance vs CA3 SG (all E4 with CA3)")
r1b = A.corr_report(d.rate_5sd, d["sgpool_ca3_5sd_30-50"], "abundance vs CA3 SG pooled")
d2 = MWM1[MWM1.genotype == "E4"].copy(); d2 = d2.join(C1[["rate_5sd"]], how="inner")
rs = []
for pop in ("e4", "all"):
    base = MWM1.copy(); base["x"] = 1.0
    ls = A.scores(base, A.MWM_LEARN, A.MWM_LEARN_INV, pop); ms = A.scores(base, A.MWM_MEM, A.MWM_MEM_INV, pop)
    e4 = base.genotype == "E4"
    rs.append(A.corr_report(ls[e4], ms[e4], f"learning vs memory precision score, E4 mice recorded, z pop {pop}"))
cnt = 0
for a in A.MWM_LEARN:
    for b in A.MWM_MEM:
        cnt += stats.pearson(d2[a], d2[b])["p"] < 0.05
print("variant: significant among the 20 learning x probe metric correlations:", cnt)
out = "reproduced" if (r1["p"] >= 0.05 and rs[0]["p"] >= 0.05) else "not_reproduced"
record("screen-features-independent", out, measured=[round(r1["r"], 3), round(rs[0]["r"], 3)])


**Result:** Not reproduced. SWR abundance and CA3 SG power are correlated in the 16 E4 mice, R = -0.533, p = 0.034 (pooled: R = -0.543, p = 0.030); the learning and memory precision scores are not (R = -0.144, p = 0.60). The paper's test was on 13 mice with CA3 sites, which the data's labels do not identify.

## fig2n-rep-learning-score

> In the replication cohort, the learning performance score predicted from SWR abundance by the screen cohort's model captures 56% of the variance of the actual score.

In [ ]:
def pred_claim(cid, train, test, er2, tol, inputs):
    res = {}
    for pop in ("e4", "all"):
        tr, te = train(pop), test(pop)
        p = stats.predict_scores(tr.x.values, tr.score.values, te.x.values, te.score.values)
        print(f"{inputs}, z pop {pop}: n test {p['n']}, R = {p['r']:+.3f}, R2 = {p['r2']:.3f}, F(1,{p['df'][1]}) = {p['F']:.2f}, p = {p['p']:.4f}; out-of-sample F = {p['F_oos']:.2f}, p = {p['p_oos']:.3f}")
        res[pop] = p
    ok = [r["r"] > 0 and r["p"] < 0.05 and abs(r["r2"] - er2) <= tol for r in res.values()]
    part = [r["r"] > 0 and r["p"] < 0.05 for r in res.values()]
    out = "reproduced" if any(ok) else ("partially_reproduced" if any(part) else "not_reproduced")
    return out, [round(res[p]["r2"], 3) for p in ("e4", "all")]
o, m = pred_claim("fig2n", lambda pop: score_table(C1, MWM1, "rate_5sd", A.MWM_LEARN, A.MWM_LEARN_INV, pop),
                  lambda pop: score_table(R13, MWM2, "rate_5sd", A.MWM_LEARN, A.MWM_LEARN_INV, pop), 0.56, 0.056, "abundance")
record("fig2n-rep-learning-score", o, measured=m)


**Result:** Not reproduced. The predicted learning score explains 1.7% of the variance of the actual score in the 11 replication E4 mice (R = +0.131, p = 0.70), against 56%.

## fig2r-rep-memory-score

> In the replication cohort, the memory precision score predicted from CA3 SG power by the screen cohort's model captures 79% of the variance of the actual score.

In [ ]:
outs = {}
for sgc in SG_READINGS.values():
    o, m = pred_claim("fig2r", lambda pop: score_table(C1, MWM1, sgc, A.MWM_MEM, A.MWM_MEM_INV, pop),
                      lambda pop: score_table(R13, MWM2, sgc, A.MWM_MEM, A.MWM_MEM_INV, pop), 0.79, 0.079, sgc)
    outs[sgc] = (o, m)
o = outs[SG][0]
record("fig2r-rep-memory-score", o, assumed=len({v[0] for v in outs.values()}) > 1, measured=outs[SG][1], readings={k: v[0] for k, v in outs.items()})


**Result:** Not reproduced. The predicted memory precision score explains 2.0 to 3.2% of the variance across readings (R = +0.142, p = 0.74 per session; n = 8), against 79%.

## rep-all-replicated

> All the predictive relationships found in the screen cohort remain significant in the replication cohort, more than chance would give.

In [ ]:
ids = ["rep-mwm-swr-slope-days-1-2", "rep-mwm-swr-latency-day-3", "rep-mwm-swr-overnight", "rep-mwm-swr-slope-days-1-3",
       "rep-mwm-ca3-probe1-quadrant", "rep-mwm-ca3-probe1-crossings", "rep-mwm-ca3-probe1-auc"]
held = [OUT[i]["outcome"] in ("reproduced", "partially_reproduced") for i in ids]
# rep-mwm-ca3-probe2 holds two relationships
for k_, (xc, yc, er) in {"q": ("pct_quadrant_p2", 1, 0.655), "a": ("auc_p2", -1, -0.713)}.items():
    d = join(R13, MWM2, [SG]); r = stats.pearson(d[SG], d[xc]); held.append(bool(np.sign(r["r"]) == np.sign(er) and r["p"] < 0.05))
k = sum(held); print("relationships holding (sign and unadjusted p < 0.05):", k, "of", len(held), "; one-sided binomial p =", stats.binom_greater(k, len(held)))
out = "reproduced" if k == 9 and stats.binom_greater(k, 9) < 0.001 else ("partially_reproduced" if k >= 7 else "not_reproduced")
record("rep-all-replicated", out, measured=f"{k} of 9")


**Result:** Not reproduced. None of the 9 replication relationships has the paper's sign with p < 0.05.

## cohort-swr-differences

> The two cohorts differ in mean SWR abundance (0.12 against 0.20 Hz) and in mean SWR baseline (16.2 against 11.5 uV), though the predictive relationships hold in both.

In [ ]:
a, b = C1[C1.genotype == "E4"], R13[R13.genotype == "E4"]
r = stats.ttest(a.rate_5sd, b.rate_5sd); print(f"abundance, E4 screen vs E4 replication 13-17 mo: means {r['mean_a']:.3f} vs {r['mean_b']:.3f} Hz, t({r['df']}) = {r['t']:.2f}, p = {r['p']:.2g}")
base = {}
for name, (x, y) in {"(a) E4 both cohorts": (a, b), "(b) all mice both cohorts": (C1, R13),
                     "(c) E4 with CA3 sites": (a[a[SG].notna()], b[b[SG].notna()])}.items():
    rb = stats.ttest(x.base, y.base); base[name] = rb
    print(f"baseline {name}: means {rb['mean_a']:.2f} vs {rb['mean_b']:.2f} (int16 units), t({rb['df']}) = {rb['t']:.2f}, p = {rb['p']:.2g}")
okab = abs(r["mean_a"] - 0.12) <= 0.012 and abs(r["mean_b"] - 0.20) <= 0.02 and r["p"] < 1e-4
okb = {k_: abs(v["mean_a"] - 16.2) <= 1.62 and abs(v["mean_b"] - 11.5) <= 1.15 and v["p"] < 1e-4 for k_, v in base.items()}
dirb = {k_: v["mean_a"] > v["mean_b"] and v["p"] < 0.05 for k_, v in base.items()}
print("abundance within tolerance and p<1e-4:", okab, "; baseline per reading:", okb, "; direction with p<0.05:", dirb)
out = "reproduced" if okab and any(okb.values()) else ("partially_reproduced" if r["mean_a"] < r["mean_b"] and r["p"] < 0.05 and any(dirb.values()) else "not_reproduced")
record("cohort-swr-differences", out, measured=[round(r["mean_a"], 3), round(r["mean_b"], 3), round(base["(a) E4 both cohorts"]["mean_a"], 2), round(base["(a) E4 both cohorts"]["mean_b"], 2)])


**Result:** Reproduced, under reading (b) for the baseline. SWR abundance is 0.113 Hz in screen E4 mice against 0.197 Hz in replication E4 mice, t(29) = -4.52, p = 9.7e-05 (the paper's t(29) = 4.75). The baseline of all mice is 16.18 against 11.43 int16 units, t(51) = 5.39, p = 1.8e-06, within tolerance of 16.2 and 11.5; the E4-only readings give 16.17 against 9.57 (t(29)) and 16.17 against 9.77 (t(25), reading (c)), outside tolerance for the second value. The units are taken as microvolts, as the paper prints them (as mV), which the data cannot confirm.

## robust-detection-parameters

> In aged apoE4-KI mice of the screen cohort, the predictive relationships hold whether SWRs are detected at 150-250 or 125-250 Hz and at a 3 or 5 SD threshold.

In [ ]:
rels = [("rate", m, s) for m, s in (("slope_1_2", -1), ("slope_1_3", -1), ("latency_d3", -1), ("night_1", -1))] + \
       [("sg", m, s) for m, s in (("pct_quadrant_p1", 1), ("pct_quadrant_p2", 1), ("crossings_p1", 1), ("auc_p1", -1), ("auc_p2", -1))]
cols = {"rate": {"150-250, 5 SD": "rate_5sd", "150-250, 3 SD": "rate_3sd", "125-250, 5 SD": "rate_125_5sd", "125-250, 3 SD": "rate_125_3sd"},
        "sg": {"150-250, 5 SD": "sg_ca3_5sd_30-50", "150-250, 3 SD": "sg_ca3_3sd_30-50"}}
held = 0; tot = 0; rows = []
for kind, m, s in rels:
    for name, c in cols[kind].items():
        d = join(C1, MWM1, [c]); r = stats.pearson(d[c], d[m]); h = np.sign(r["r"]) == s and r["p"] < 0.05
        rows.append((kind, m, name, r["n"], round(r["r"], 3), round(r["p"], 4), h)); held += h; tot += 1
print(pd.DataFrame(rows, columns=["property", "metric", "detection", "n", "R", "p", "holds"]).to_string())
print(f"held: {held} of {tot}. The SG power of SWRs detected with the 125-250 Hz filter was not computed (18 of the 36 correlations).")
record("robust-detection-parameters", "blocked" if False else ("reproduced" if held == tot and tot == 36 else ("partially_reproduced" if held >= 0.89 * tot else "not_reproduced")), measured=f"{held} of {tot} run")


**Result:** Not reproduced. Of the 26 of 36 correlations run, 8 have the expected sign with p < 0.05; the 10 not run (CA3 SG power of SWRs detected at 125-250 Hz) could not lift the count to the 32 needed.

## age-no-correlation

> Age within the cohort does not significantly correlate with SWR properties or MWM performance.

In [ ]:
res = {}
for name, E, B in (("(a) screen", C1, MWM1), ("(b) replication 13-17 mo", R13, MWM2)):
    d = E[E.genotype == "E4"][["age_months", "rate_5sd", SG]].join(B.drop(columns=["genotype"]), how="left")
    n_sig = 0; lines = []
    for c in ["rate_5sd", SG] + A.MWM_LEARN + A.MWM_MEM:
        r = stats.pearson(d.age_months, d[c]); n_sig += r["p"] < 0.05; lines.append(f"{c}: n {r['n']}, R {r['r']:+.2f}, p {r['p']:.3f}")
    res[name] = n_sig; print(name, "age range", d.age_months.min(), d.age_months.max(), "; significant:", n_sig); print("   " + "; ".join(lines))
out = "reproduced" if any(v == 0 for v in res.values()) else "not_reproduced"
record("age-no-correlation", out, measured=str(res))


**Result:** Reproduced. Age does not correlate with SWR abundance, CA3 SG power or the nine MWM score metrics in the E4 mice of either cohort (0 of 11 with p < 0.05 in each).

## figs2-nonspatial-no-difference

> In the replication cohort there are no genotype differences in open field, elevated plus maze, hot plate or visible-platform MWM performance.

In [ ]:
of, ep, hp, vis = behavior.open_field(), behavior.elevated_plus(), behavior.hot_plate(), behavior.mwm_visible(2, "14-18mo")
hp = hp[~hp.excluded.astype(bool)]
tests = [(of, "pct_center", "t"), (of, "fine_movements", "t"), (of, "ambulatory_movements", "t"), (of, "total_movements", "t"),
         (ep, "pct_time_open", "t"), (ep, "dist_open_in", "mw"), (ep, "dist_closed_in", "t"), (ep, "dist_total_in", "t"),
         (hp, "latency_s", "mw"), (vis, "visible_latency_s", "t")]
nsig = 0
for df_, c, t in tests:
    a, b = df_[df_.genotype == "E3"][c], df_[df_.genotype == "E4"][c]
    r = stats.ttest(a, b) if t == "t" else stats.mannwhitney(a, b)
    s = f"t({r['df']}) = {r['t']:.2f}" if t == "t" else f"U = {r['U']:.1f}"
    print(f"{c:22s} n {a.notna().sum()}/{b.notna().sum()}: {s}, p = {r['p']:.3f}; E3 mean {a.mean():.2f}, E4 mean {b.mean():.2f}")
    nsig += r["p"] < 0.05
record("figs2-nonspatial-no-difference", "reproduced" if nsig == 0 else "not_reproduced", measured=f"{nsig} of 10 tests with p < 0.05")
NONSP = {"pct_center": of, "fine_movements": of, "ambulatory_movements": of, "total_movements": of, "pct_time_open": ep,
         "dist_open_in": ep, "dist_closed_in": ep, "dist_total_in": ep, "latency_s": hp, "visible_latency_s": vis}


**Result:** Reproduced. None of the 10 tests has p < 0.05 (smallest p = 0.118, visible-platform latency). The hot plate has 12 E3 and 10 E4 mice in the data set, against 12 and 13 in the legend.

## nonspatial-not-correlated

> None of the non-spatial behaviors correlates significantly with spatial task performance or with SWR properties.

In [ ]:
base = R13[R13.genotype == "E4"][["rate_5sd", SG]]
mw = MWM2.copy(); mw["x"] = 1.0; mw["learn"] = A.scores(mw, A.MWM_LEARN, A.MWM_LEARN_INV, "e4"); mw["mem"] = A.scores(mw, A.MWM_MEM, A.MWM_MEM_INV, "e4")
ap = APA_W.copy(); ap["x"] = 1.0
ap["alearn"] = A.scores(ap, ["latency_first_entrance_s_d2", "path_length_m_d2", "pct_time_opp_d2"], [], "e4")
ap["amem"] = A.scores(ap, ["entrances_d2", "dpb_d2", "dpb_d3"], ["entrances_d2", "dpb_d2", "dpb_d3"], "e4")
nsig = 0; ntot = 0; sig = []
for c, df_ in NONSP.items():
    v = df_.dropna(subset=["name"]).copy(); v["key"] = v.name.map(behavior.norm_name); v = v[v.genotype == "E4"].set_index("key")[c]
    for tname, t in (("rate_5sd", base.rate_5sd), ("CA3", base[SG]), ("MWM learn", mw.learn[mw.genotype == "E4"]), ("MWM mem", mw.mem[mw.genotype == "E4"]),
                     ("APA learn", ap.alearn[ap.genotype == "E4"]), ("APA mem", ap.amem[ap.genotype == "E4"])):
        j = pd.concat([v, t], axis=1, join="inner").dropna()
        r = stats.pearson(j.iloc[:, 0], j.iloc[:, 1]); ntot += 1
        if r["p"] < 0.05: nsig += 1; sig.append((c, tname, r["n"], round(r["r"], 2), round(r["p"], 3)))
print("significant:", nsig, "of", ntot, "(expected by chance about", round(0.05 * ntot, 1), ")", sig)
record("nonspatial-not-correlated", "reproduced" if nsig == 0 else "not_reproduced", measured=f"{nsig} of {ntot}")


**Result:** Not reproduced on the claim's test. 2 of 60 correlations have p < 0.05 (closed-arm distance against SWR abundance, R = -0.60, p = 0.050 with n = 11; hot plate latency against CA3 SG power, R = -0.91, n = 5), fewer than the 3 expected by chance.

## ca1-dg-not-predictive

> SWR-associated SG power in CA1 stratum radiatum and in the dentate gyrus does not consistently and significantly predict memory performance in apoE4-KI mice.

In [ ]:
t = T88[T88.property.isin(["CA1-sr", "DG"])]
print("n per property:", t.groupby("property").n.max().to_dict()); print("significant:"); print(t[t.p < 0.05].round(3).to_string())
k = int((t.p < 0.05).sum())
record("ca1-dg-not-predictive", "reproduced" if k == 0 else "not_reproduced", measured=f"{k} of 44 with p < 0.05")


**Result:** Not reproduced on the claim's test. 1 of the 44 correlations has p < 0.05 (DG SG power against day 1 latency, R = -0.508, p = 0.045), where the claim requires none. No CA1-sr correlation is significant.

## e3-relationships

> In apoE3-KI mice, SWR abundance predicts escape latency on hidden day 2 in both cohorts, while it does not predict day 3 latency, whose relationship holds in apoE4-KI mice; apoE3-KI SWR abundance does not overlap that of apoE4-KI mice.

In [ ]:
d1 = join(C1, MWM1, ["rate_5sd"], "E3"); d2 = join(R13, MWM2, ["rate_5sd"], "E3")
r1 = A.corr_report(d1.rate_5sd, d1.latency_d2, "screen E3, day 2"); r2 = A.corr_report(d2.rate_5sd, d2.latency_d2, "replication E3, day 2")
r3 = A.corr_report(d1.rate_5sd, d1.latency_d3, "screen E3, day 3")
e3min, e4max = C1[C1.genotype == "E3"].rate_5sd.min(), C1[C1.genotype == "E4"].rate_5sd.max()
print(f"non-overlap: E3 minimum {e3min:.3f} vs E4 maximum {e4max:.3f} -> {'no overlap' if e3min > e4max else 'overlap'}")
h1 = A.judge_corr(r1, -0.6, 0.06) != "not_reproduced"; h2 = A.judge_corr(r2, -0.746, 0.075) != "not_reproduced"; h3 = r3["p"] >= 0.05
out = "reproduced" if (h1 and h2 and h3) else ("partially_reproduced" if (h1 or h2) else "not_reproduced")
record("e3-relationships", out, measured=[round(r1["r"], 3), round(r2["r"], 3)])


**Result:** Not reproduced. In screen E3 mice, day 2 latency against SWR abundance gives R = -0.548, p = 0.053 (n = 13); in the 6 replication E3 mice linked to behavior, R = +0.407, p = 0.42. The day 3 correlation is absent (R = -0.015, p = 0.96), and E3 and E4 abundance overlap (E3 minimum 0.194, E4 maximum 0.210 Hz).

## fig3b-apa-entrances-day1

> Aged apoE4-KI mice make more entrances into the APA shock zone than apoE3-KI mice on training day 1.

In [ ]:
a = TAB["APA"]
for k in range(1, 5):
    r = stats.ttest(a[a.genotype == "E4"][f"entrances_d{k}"], a[a.genotype == "E3"][f"entrances_d{k}"])
    print(f"day {k}: E4 {r['mean_a']:.1f} vs E3 {r['mean_b']:.1f}, t({r['df']}) = {r['t']:.2f}, p = {r['p']:.4f}, Sidak (4 days) {stats.sidak(r['p'], 4):.4f}")
    if k == 1: r1 = r
padj = stats.sidak(r1["p"], 4)
out = "reproduced" if r1["mean_a"] > r1["mean_b"] and padj < 0.05 else ("partially_reproduced" if r1["mean_a"] > r1["mean_b"] and r1["p"] < 0.05 else "not_reproduced")
record("fig3b-apa-entrances-day1", out, measured=f"t({r1['df']}) = {r1['t']:.2f}, adjusted p = {padj:.4f}")


**Result:** Reproduced. E4 mice entered the shock zone more on day 1, 14.5 against 8.2, t(23) = 3.43, Sidak-adjusted p = 0.0091, as printed.

## figs3-apa-day1-and-probe

> On APA day 1 apoE4-KI mice travel less, spend less time opposite the shock zone and move further from it per bout, and on the probe they receive more pseudoshocks per entrance than apoE3-KI mice.

In [ ]:
a = TAB["APA"]; aa = A.apa_metrics("dist_per_bout_cm")
e3, e4 = a[a.genotype == "E3"], a[a.genotype == "E4"]
held = 0
r = stats.ttest(e4.path_length_m_d1, e3.path_length_m_d1); h = r["t"] < 0 and r["p"] < 0.05; held += h; print(f"path length d1: E4 {r['mean_a']:.1f} vs E3 {r['mean_b']:.1f} m, t({r['df']}) = {r['t']:.2f}, p = {r['p']:.4f}, holds {h}")
r = stats.ttest(e4.pct_time_opp_d1, e3.pct_time_opp_d1); h = r["t"] < 0 and r["p"] < 0.05; held += h; print(f"% time opposite d1: E4 {r['mean_a']:.1f} vs E3 {r['mean_b']:.1f}, t({r['df']}) = {r['t']:.2f}, p = {r['p']:.4f}, holds {h}")
for nm, df_ in (("walk bouts", a), ("all bouts", aa)):
    x4, x3 = df_[df_.genotype == "E4"].dpb_d1, df_[df_.genotype == "E3"].dpb_d1
    r = stats.mannwhitney(x4, x3); h = x4.median() > x3.median() and r["p"] < 0.05
    print(f"distance per bout d1 ({nm}): E4 median {x4.median():.2f} vs E3 {x3.median():.2f} cm, U = {r['U']:.0f}, p = {r['p']:.4f}, E4 further and p<0.05: {h}")
    if nm == "walk bouts": held += h
ap = behavior.apa(); pr = ap[(ap.session == "probe") & ~ap.excluded.astype(bool)]
x4, x3 = pr[pr.genotype == "E4"].pseudoshocks_per_entrance, pr[pr.genotype == "E3"].pseudoshocks_per_entrance
r = stats.ttest(x4, x3); h = r["t"] > 0 and r["p"] < 0.05; held += h
print(f"probe pseudoshocks per entrance: E4 {r['mean_a']:.2f} vs E3 {r['mean_b']:.2f}, t({r['df']}) = {r['t']:.2f}, p = {r['p']:.4f}, holds {h}")
out = "reproduced" if held == 4 else ("partially_reproduced" if held >= 2 else "not_reproduced")
record("figs3-apa-day1-and-probe", out, measured=f"{held} of 4")


**Result:** Partially reproduced. Path length (t(23) = -4.64), percent time opposite (t(23) = -2.27) and probe pseudoshocks per entrance (t(18) = 2.76) reproduce the printed statistics with the stated directions. Distance per bout reproduces the printed U = 30 (p = 0.0098, walk bouts), but E4 mice moved less far per bout than E3 mice (median 1.37 against 3.80 cm), against the Results text.

## apa-40-comparisons

> Of 40 comparisons of 5 APA metrics over 4 days against 2 SWR properties, 8 are significant at alpha 0.05, more than chance, most of them on day 2.

In [ ]:
rows = []
for pc in ("rate_5sd", SG):
    d = join(R13, APA_W, [pc])
    for m in A.APA5:
        for k in range(1, 5):
            r = stats.pearson(d[pc], d[f"{m}_d{k}"]); rows.append((pc, m, k, r["n"], round(r["r"], 3), r["p"]))
T40 = pd.DataFrame(rows, columns=["property", "metric", "day", "n", "R", "p"])
print(T40[T40.p < 0.05].round(4).to_string()); k = int((T40.p < 0.05).sum())
byday = T40[T40.p < 0.05].groupby("day").size().reindex(range(1, 5), fill_value=0)
print("significant:", k, "of 40; binomial p =", round(stats.binom_greater(k, 40), 5), "; by day:", byday.to_dict())
ok = abs(k - 8) <= 1 and stats.binom_greater(k, 40) < 0.05 and byday[2] > byday.drop(2).max()
record("apa-40-comparisons", "reproduced" if ok else "not_reproduced", measured=f"{k} of 40; day 2: {byday[2]}")


**Result:** Not reproduced. 2 of the 40 correlations have p < 0.05 (one on day 1, one on day 2), binomial p = 0.60, against 8.

## apa-day2-variance

> Variance across mice of shock zone entrances is higher on APA day 2 than day 1 (15.76 against 33.82).

In [ ]:
a = TAB["APA"]
corrmice = join(R13, APA_W, ["rate_5sd"]).index
res = {}
for name, d in (("(a) 13 E4", a[a.genotype == "E4"]), ("(b) E4 of the correlations", APA_W.loc[APA_W.index.isin(corrmice)]), ("(c) all", a)):
    v1, v2 = d.entrances_d1.var(ddof=1), d.entrances_d2.var(ddof=1); res[name] = (v1, v2)
    print(f"{name}: n {d.entrances_d1.notna().sum()}, variance day 1 {v1:.2f}, day 2 {v2:.2f}")
ok = [abs(v1 - 15.76) <= 1.58 and abs(v2 - 33.82) <= 3.38 for v1, v2 in res.values()]
part = [v2 > v1 for v1, v2 in res.values()]
record("apa-day2-variance", "reproduced" if any(ok) else ("partially_reproduced" if any(part) else "not_reproduced"), measured=[round(x, 2) for x in res["(a) 13 E4"]])


**Result:** Partially reproduced. The day 2 variance exceeds day 1 under all three readings, but no reading gives both values within tolerance: (a) 24.77 and 47.97, (b) 7.21 and 27.07, (c) 31.18 and 57.91, against 15.76 and 33.82.

## figs3fg-day2-predicts-later

> In apoE4-KI mice, entrances on APA day 2 predict entrances on days 3 and 4, and the top 50% of day 2 performers keep learning on days 3 and 4 while the bottom 50% do not.

In [ ]:
e4 = TAB["APA"][TAB["APA"].genotype == "E4"].copy()
r3 = A.corr_report(e4.entrances_d2, e4.entrances_d3, "day 2 vs day 3"); r4 = A.corr_report(e4.entrances_d2, e4.entrances_d4, "day 2 vs day 4")
LOO["figs3fg-day2-predicts-later"] = r3; LOO["figs3fg-day2-predicts-later (day 4)"] = r4
e4 = e4.sort_values("entrances_d2"); n = len(e4); top = e4.iloc[:int(np.ceil(n / 2))]; bot = e4.iloc[int(np.ceil(n / 2)):]
print("top (fewest day 2 entrances)", len(top), "bottom", len(bot))
grp = {}
for k in (1, 3, 4):
    r = stats.ttest(top[f"entrances_d{k}"], bot[f"entrances_d{k}"]); grp[k] = stats.sidak(r["p"], 3)
    print(f"(a) day {k}: top {r['mean_a']:.1f} vs bottom {r['mean_b']:.1f}, t({r['df']}) = {r['t']:.2f}, p = {r['p']:.4f}, Sidak {grp[k]:.4f}")
L = pd.concat([pd.DataFrame({"g": g_, "day": k, "y": d_[f"entrances_d{k}"]}) for g_, d_ in (("top", top), ("bot", bot)) for k in range(1, 5)])
import statsmodels.formula.api as smf
fit = smf.ols("y ~ C(g) * C(day)", data=L).fit(); mse, dfe = fit.mse_resid, fit.df_resid
grpb = {}
for k in (1, 3, 4):
    a_, b_ = L[(L.g == "top") & (L.day == k)].y, L[(L.g == "bot") & (L.day == k)].y
    t = (a_.mean() - b_.mean()) / np.sqrt(mse * (1 / len(a_) + 1 / len(b_))); from scipy import stats as ss
    p = 2 * ss.t.sf(abs(t), dfe); grpb[k] = stats.sidak(p, 3); print(f"(b) day {k}: t({dfe:.0f}) = {t:.2f}, Sidak p = {grpb[k]:.4f}")
okc = r3["r"] > 0 and r3["p"] < 0.05 and abs(r3["r"] - 0.904) <= 0.09 and r4["r"] > 0 and r4["p"] < 0.05 and abs(r4["r"] - 0.773) <= 0.077
okg = [g[3] < 0.05 and g[4] < 0.05 and g[1] >= 0.05 for g in (grp, grpb)]
record("figs3fg-day2-predicts-later", "reproduced" if okc and any(okg) else ("partially_reproduced" if (r3["p"] < 0.05 and r4["p"] < 0.05) or any(okg) else "not_reproduced"),
       measured=[round(r3["r"], 3), round(r4["r"], 3)])


**Result:** Reproduced. Day 2 entrances predict day 3 (R = +0.904, F(1,11) = 49.27) and day 4 (R = +0.773, F(1,11) = 16.32) entrances, as printed; the top half differs from the bottom half on days 3 and 4 and not on day 1 under both readings (reading (b): t(44) = -1.66, -5.06, -5.28).

## fig3c-swr-latency-first-entrance

> In the replication cohort, SWR abundance predicts latency_first_entrance_s_d2 (expected R = 0.65).

In [ ]:
CL["fig3c-swr-latency-first-entrance"] = corr_claim("fig3c-swr-latency-first-entrance", "R13", "APA_W", "rate_5sd", "latency_first_entrance_s_d2", 0.65, 0.065)

**Result:** Not reproduced. R = +0.321, p = 0.37 (n = 10; paper R = 0.65).

## fig3de-swr-path-and-quadrant

> In the replication cohort, SWR abundance predicts path length and percent time in the quadrant opposite the shock zone on APA day 2.

In [ ]:
subs = []
corr_claim("fig3de-swr-path-and-quadrant-0", "R13", "APA_W", "rate_5sd", "path_length_m_d2", 0.82, 0.082); subs.append(OUT.pop("fig3de-swr-path-and-quadrant-0"))
corr_claim("fig3de-swr-path-and-quadrant-1", "R13", "APA_W", "rate_5sd", "pct_time_opp_d2", 0.69, 0.069); subs.append(OUT.pop("fig3de-swr-path-and-quadrant-1"))
o = [s_["outcome"] for s_ in subs]
out = "reproduced" if all(x == "reproduced" for x in o) else ("partially_reproduced" if any(x != "not_reproduced" for x in o) else "not_reproduced")
record("fig3de-swr-path-and-quadrant", out, assumed=any(s_["assumed"] for s_ in subs), measured=[s_["measured"] for s_ in subs])


**Result:** Not reproduced. Path length R = +0.277, p = 0.44; percent time opposite R = +0.280, p = 0.43 (n = 10).

## fig3g-ca3-entrances

> In the replication cohort, higher CA3 SG power during SWRs predicts fewer shock zone entrances on APA day 2, and also on days 3 and 4.

In [ ]:
subs = []
corr_claim("fig3g-ca3-entrances-0", "R13", "APA_W", "SG", "entrances_d2", -0.91, 0.091); subs.append(OUT.pop("fig3g-ca3-entrances-0"))
corr_claim("fig3g-ca3-entrances-1", "R13", "APA_W", "SG", "entrances_d3", -0.76, 0.076); subs.append(OUT.pop("fig3g-ca3-entrances-1"))
corr_claim("fig3g-ca3-entrances-2", "R13", "APA_W", "SG", "entrances_d4", -0.72, 0.072); subs.append(OUT.pop("fig3g-ca3-entrances-2"))
o = [s_["outcome"] for s_ in subs]
out = "reproduced" if all(x == "reproduced" for x in o) else ("partially_reproduced" if o[0] != "not_reproduced" and (o[1] != "not_reproduced" or o[2] != "not_reproduced") else "not_reproduced")
record("fig3g-ca3-entrances", out, assumed=any(s_["assumed"] for s_ in subs), measured=[s_["measured"] for s_ in subs])


**Result:** Not reproduced. In the 7 E4 mice with CA3 labels and APA, entrances on days 2, 3 and 4 give R = -0.591, -0.418, -0.160, none with p < 0.05.

## fig3h-ca3-distance-per-bout

> In the replication cohort, mice with lower CA3 SG power during SWRs move further from the shock zone per movement bout on APA days 2 and 3.

In [ ]:
subs = []
corr_claim("fig3h-ca3-distance-per-bout-0", "R13", "APA_W", "SG", "dpb_d2", -0.75, 0.075); subs.append(OUT.pop("fig3h-ca3-distance-per-bout-0"))
corr_claim("fig3h-ca3-distance-per-bout-1", "R13", "APA_W", "SG", "dpb_d3", -0.7, 0.07); subs.append(OUT.pop("fig3h-ca3-distance-per-bout-1"))
o = [s_["outcome"] for s_ in subs]
out = "reproduced" if all(x == "reproduced" for x in o) else ("partially_reproduced" if any(x != "not_reproduced" for x in o) else "not_reproduced")
record("fig3h-ca3-distance-per-bout", out, assumed=any(s_["assumed"] for s_ in subs), measured=[s_["measured"] for s_ in subs])


In [ ]:
print("variant: distance per bout over all bouts (pull_apa_metrics.py first column)")
for k in (2, 3):
    d = join(R13, APA_ALL, [SG]); A.corr_report(d[SG], d[f"dpb_d{k}"], f"all bouts, day {k}")

**Result:** Not reproduced. Distance per bout (walk bouts) gives R = -0.197 (day 2) and -0.472 (day 3), n = 7, p > 0.28; over all bouts, day 3 gives R = -0.701, p = 0.080.

## fig3f-apa-learning-score

> The APA learning performance score predicted from SWR abundance by the screen cohort's model accounts for 75% of the variance in the actual score.

In [ ]:
res = {}
for mname, mets in (("text: path, % opposite", ["path_length_m_d2", "pct_time_opp_d2"]), ("Methods: + latency to first entrance", ["latency_first_entrance_s_d2", "path_length_m_d2", "pct_time_opp_d2"])):
    print(mname); res[mname] = pred_claim("fig3f", lambda pop: score_table(C1, MWM1, "rate_5sd", A.MWM_LEARN, A.MWM_LEARN_INV, pop),
                                   lambda pop: score_table(R13, APA_W, "rate_5sd", mets, [], pop), 0.75, 0.075, "abundance -> APA learning")
o = [v[0] for v in res.values()]
out = "reproduced" if "reproduced" in o else ("partially_reproduced" if "partially_reproduced" in o else "not_reproduced")
record("fig3f-apa-learning-score", out, measured=res["Methods: + latency to first entrance"][1], readings={k: v[0] for k, v in res.items()})


**Result:** Not reproduced. The predicted APA learning score explains 10.7 to 12.6% of the variance across readings (n = 10, p >= 0.31), against 75%.

## fig3i-apa-memory-score

> The APA memory precision score predicted from CA3 SG power by the screen cohort's model accounts for 73% of the variance in the actual score.

In [ ]:
outs = {}
mets = ["entrances_d2", "dpb_d2", "dpb_d3"]
for sgc in SG_READINGS.values():
    outs[sgc] = pred_claim("fig3i", lambda pop: score_table(C1, MWM1, sgc, A.MWM_MEM, A.MWM_MEM_INV, pop),
                           lambda pop: score_table(R13, APA_W, sgc, mets, mets, pop), 0.73, 0.073, sgc)
record("fig3i-apa-memory-score", outs[SG][0], assumed=len({v[0] for v in outs.values()}) > 1, measured=outs[SG][1], readings={k: v[0] for k, v in outs.items()})


**Result:** Not reproduced. The predicted APA memory precision score explains 40 to 41% of the variance (R = +0.64, p = 0.12, n = 7), against 73%.

## figs3ij-mwm-predicts-apa

> In apoE4-KI mice, MWM performance scores predict the corresponding APA scores, learning against learning and memory precision against memory precision.

In [ ]:
res = {}
ap = APA_W.copy(); mw = MWM2.copy()
for pop in ("e4", "all"):
    ap["x"] = 1.0; mw["x"] = 1.0
    ml = A.scores(mw, A.MWM_LEARN, A.MWM_LEARN_INV, pop); mm = A.scores(mw, A.MWM_MEM, A.MWM_MEM_INV, pop)
    for lname, lm in (("2 metrics", ["path_length_m_d2", "pct_time_opp_d2"]), ("3 metrics", ["latency_first_entrance_s_d2", "path_length_m_d2", "pct_time_opp_d2"])):
        al = A.scores(ap, lm, [], pop); am = A.scores(ap, ["entrances_d2", "dpb_d2", "dpb_d3"], ["entrances_d2", "dpb_d2", "dpb_d3"], pop)
        rec = R13[(R13.genotype == "E4")].index   # E4 mice of the correlations: recorded at 13-17 months
        j = pd.DataFrame({"ml": ml[mw.genotype == "E4"], "mm": mm[mw.genotype == "E4"]}).join(pd.DataFrame({"al": al[ap.genotype == "E4"], "am": am[ap.genotype == "E4"]}), how="inner")
        rec3 = R13[(R13.genotype == "E4") & R13[SG].notna()].index   # and with CA3 sites, for memory precision
        jl, jm = j[j.index.isin(rec)], j[j.index.isin(rec3)]
        r1 = A.corr_report(jl.ml, jl.al, f"z pop {pop}, APA learning {lname}: learning, E4 recorded at 13-17 mo")
        r2 = A.corr_report(jm.mm, jm.am, f"z pop {pop}: memory precision, E4 recorded at 13-17 mo with CA3 sites")
        res[(pop, lname)] = (r1, r2)
        A.corr_report(j.ml, j.al, f"variant, all {len(j)} E4 mice with both tasks: learning"); A.corr_report(j.mm, j.am, "variant, all E4 with both tasks: memory")
def h(r, er, tol): return r["r"] > 0 and r["p"] < 0.05 and abs(r["r"] - er) <= tol
both = [h(a, 0.665, 0.067) and h(b, 0.783, 0.078) for a, b in res.values()]
one = [h(a, 0.665, 0.067) or h(b, 0.783, 0.078) for a, b in res.values()]
k0 = ("e4", "3 metrics")
record("figs3ij-mwm-predicts-apa", "reproduced" if any(both) else ("partially_reproduced" if any(one) else "not_reproduced"), measured=[round(res[k0][0]["r"], 3), round(res[k0][1]["r"], 3)])


**Result:** Partially reproduced. The learning scores correlate in the 10 E4 mice recorded at 13-17 months with the two-metric APA learning score (R = +0.638, p = 0.047; +0.658, p = 0.039 with the all-mice z-score), within tolerance of 0.665; with the three-metric score R = +0.562, p = 0.091. The memory precision scores, in the 7 E4 mice with CA3 labels, give R = +0.626, p = 0.13. With all 11 E4 mice that did both tasks (not the legend's mice) memory precision gives R = +0.687, p = 0.019.

## figs4ab-young-deficits

> In the replication cohort, apoE4-KI mice have lower SWR abundance and lower CA3 SG power during SWRs than apoE3-KI mice already at 5-8 months and at every age tested.

In [ ]:
R2 = MICE[MICE.cohort == 2].copy()
ok = {}
for c in ("rate_5sd", SG):
    d = R2[["key", "genotype", "age_group", c]].dropna()
    art = stats.art_anova_rm(d, "key", "genotype", "age_group", c)["between"]
    lower = d[d.genotype == "E4"][c].mean() < d[d.genotype == "E3"][c].mean()
    print(f"{c}: genotype effect ({art['path']}) F = {art['F']:.2f}, df {art['df']}, p = {art['p']:.2g}, n mice {art['n_subjects']}, E4 lower: {lower}")
    young = None
    for ag in ("5-8mo", "9-11mo", "13-17mo"):
        a, b = d[(d.age_group == ag) & (d.genotype == "E3")][c], d[(d.age_group == ag) & (d.genotype == "E4")][c]
        r = stats.mannwhitney(a, b); padj = stats.sidak(r["p"], 3)
        print(f"   {ag}: n {len(a)}/{len(b)}, U = {r['U']:.0f}, p = {r['p']:.4f}, Sidak (3 ages) {padj:.4f}, E3 median {a.median():.3f}, E4 median {b.median():.3f}")
        if ag == "5-8mo": young = b.median() < a.median() and padj < 0.05
    ok[c] = (art["p"] < 0.05 and lower, young)
print(ok)
out = "reproduced" if all(a and y for a, y in ok.values()) else ("partially_reproduced" if all(a for a, _ in ok.values()) else "not_reproduced")
record("figs4ab-young-deficits", out, measured=str(ok))


**Result:** Not reproduced. SWR abundance is lower in E4 mice (mixed-effects genotype F = 85.4, p = 3.8e-10; 5-8 months U = 22, adjusted p = 0.0027), but for CA3 SG power the genotype effect is F = 3.18, p = 0.087, with E4 lower at 5-8 months (adjusted p = 0.032) and not at 9-11 or 13-17 months.

## figs4ab-aging-increase

> Over aging, SWR abundance increases in apoE3-KI mice, and CA3 SG power during SWRs increases in both genotypes.

In [ ]:
def age_test(c, g, tests=3):
    a, b = R5[R5.genotype == g][c].dropna(), R13[R13.genotype == g][c].dropna()
    r = stats.ttest(b, a); return r, stats.sidak(r["p"], tests), stats.sidak(r["p"], 1)
held = {}
for c, g in (("rate_5sd", "E3"), (SG, "E3"), (SG, "E4")):
    r, pa, pb = age_test(c, g)
    print(f"{c} {g}: 5-8 mo {r['mean_b']:.3f} -> 13-17 mo {r['mean_a']:.3f}, t({r['df']}) = {r['t']:.2f}, p = {r['p']:.4f}; Sidak (a) 3 tests {pa:.4f}, (b) unadjusted {pb:.4f}")
    held[(c, g)] = (r["t"] > 0 and pa < 0.05, r["t"] > 0 and pb < 0.05)
na = sum(v[0] for v in held.values()); nb = sum(v[1] for v in held.values())
out = "reproduced" if max(na, nb) == 3 else ("partially_reproduced" if max(na, nb) == 2 else "not_reproduced")
record("figs4ab-aging-increase", out, measured=f"{na} of 3 under reading (a), {nb} under (b)")


**Result:** Partially reproduced. E3 SWR abundance rises from 0.251 to 0.347 Hz (adjusted p = 0.0020); E4 CA3 SG power rises from 0.617 to 0.875 (p = 0.041, adjusted 0.117); E3 CA3 SG power does not change (p = 0.95, 4 mice with CA3 labels at 13-17 months). Two of three increases hold under reading (b), one under (a).

## figs4cd-baseline-sd-swr-band

> Baseline in the SWR frequency band falls over aging in both genotypes, and its SD falls in apoE4-KI but not apoE3-KI mice.

In [ ]:
res = {}
for c, g in (("base", "E3"), ("base", "E4"), ("sd", "E4"), ("sd", "E3")):
    r, pa, pb = age_test(c, g); res[(c, g)] = (r, pa)
    print(f"{c} {g}: 5-8 mo {r['mean_b']:.2f} -> 13-17 mo {r['mean_a']:.2f}, t({r['df']}) = {r['t']:.2f}, p = {r['p']:.4f}, Sidak (3) {pa:.4f}")
dec = [res[k][0]["t"] < 0 and res[k][1] < 0.05 for k in (("base", "E3"), ("base", "E4"), ("sd", "E4"))]
e3sd = res[("sd", "E3")][1] >= 0.05
out = "reproduced" if all(dec) and e3sd else ("partially_reproduced" if sum(dec) >= 2 else "not_reproduced")
record("figs4cd-baseline-sd-swr-band", out, measured=f"{sum(dec)} of 3 decreases; E3 SD no change: {e3sd}")


**Result:** Reproduced. The baseline falls in E3 (19.10 to 14.54, adjusted p = 0.020) and E4 mice (17.06 to 9.57, p < 1e-4), the SD falls in E4 mice (14.05 to 9.06, p < 1e-4) and not in E3 mice (adjusted p = 1.0).

## figs4ef-ca3-sg-baseline-stable

> Baseline and SD in the SG band in CA3 do not change over aging in either genotype.

In [ ]:
nsig = 0
for c in ("ca3_sg_base", "ca3_sg_sd"):
    for g in ("E3", "E4"):
        a, b = R5[R5.genotype == g][c].dropna(), R13[R13.genotype == g][c].dropna()
        r = stats.ttest(b, a) if g == "E3" else stats.mannwhitney(b, a); pa = stats.sidak(r["p"], 3)
        s = f"t({r['df']}) = {r['t']:.2f}" if g == "E3" else f"U = {r['U']:.0f}"
        print(f"{c} {g}: n {len(a)}/{len(b)}, 5-8 mo {a.mean():.4g} -> 13-17 mo {b.mean():.4g}, {s}, p = {r['p']:.3f}, Sidak {pa:.3f}"); nsig += pa < 0.05
record("figs4ef-ca3-sg-baseline-stable", "reproduced" if nsig == 0 else "not_reproduced", measured=f"{nsig} of 4 with adjusted p < 0.05")


**Result:** Not reproduced on the claim's test. In E4 mice the CA3 SG baseline (U = 11, adjusted p = 0.002) and SD (U = 28, adjusted p = 0.038) fall from 5-8 to 13-17 months; in E3 mice neither changes (adjusted p 0.89 and 1.0, with 4 mice at 13-17 months).

## figs4g-young-no-deficit

> Young (6-9 month) apoE4-KI mice show no MWM learning deficit.

In [ ]:
d = TAB["Cohort 2 MWM 6-9 mo"]
long = d.melt(id_vars=["mouse", "genotype"], value_vars=[f"latency_d{k}" for k in range(1, 6)], var_name="day", value_name="lat").dropna()
long["day"] = long.day.str[-1].astype(int)
art = stats.art_anova_rm(long, "mouse", "genotype", "day", "lat")["between"]
print({k: art[k] for k in ("path", "n_subjects", "F", "df", "p")}); print(long.groupby(["genotype", "day"]).lat.mean().unstack().round(1))
record("figs4g-young-no-deficit", "reproduced" if art["p"] >= 0.05 else "not_reproduced", measured=f"F = {art['F']:.2f}, p = {art['p']:.3f}")


**Result:** Reproduced. In the 26 mice at 6-9 months the genotype effect is F(1,24) = 0.08, p = 0.78.

## fig4bc-swr-not-stable

> SWR abundance is not significantly correlated across ages within mice, in either genotype.

In [ ]:
nsig = 0
for g, f in (("E3", stats.pearson), ("E4", stats.spearman)):
    for other in (R5, R13):
        j = R9[R9.genotype == g][["rate_5sd"]].join(other[other.genotype == g][["rate_5sd"]], rsuffix="_o", how="inner")
        r = f(j.rate_5sd, j.rate_5sd_o); nsig += r["p"] < 0.05
        print(g, f.__name__, "9-11 vs", other.age_group.iloc[0], "n", r["n"], "stat", round(r.get("r", r.get("rho")), 3), "p", round(r["p"], 4))
record("fig4bc-swr-not-stable", "reproduced" if nsig == 0 else "not_reproduced", measured=f"{nsig} of 4 with p < 0.05")


**Result:** Reproduced. None of the four correlations across ages has p < 0.05 (smallest p = 0.096, E3 9-11 against 5-8 months).

## fig4de-ca3-sg-stable

> CA3 SG power during SWRs at 9-11 months correlates with that at 5-8 and at 13-17 months in the same mice, in both genotypes.

In [ ]:
outs = {}
for sgc in SG_READINGS.values():
    rs = []
    for g, other, er in (("E3", R5, 0.775), ("E3", R13, 0.935), ("E4", R5, 0.619), ("E4", R13, 0.753)):
        j = R9[R9.genotype == g][[sgc]].join(other[other.genotype == g][[sgc]], rsuffix="_o", how="inner")
        r = A.corr_report(j[sgc], j[sgc + "_o"], f"{sgc} {g} 9-11 vs {other.age_group.iloc[0]}"); rs.append((r, er))
    sig = [r["r"] > 0 and r["p"] < 0.05 for r, _ in rs]; tol = [abs(r["r"] - er) <= 0.1 * er for r, er in rs]
    outs[sgc] = ("reproduced" if all(sig) and all(tol) else ("partially_reproduced" if all(sig) or sum(sig) >= 3 else "not_reproduced"), [round(r["r"], 3) for r, _ in rs])
record("fig4de-ca3-sg-stable", outs[SG][0], assumed=len({v[0] for v in outs.values()}) > 1, measured=outs[SG][1], readings={k: v[0] for k, v in outs.items()})


**Result:** Not reproduced. The E3 correlations hold (R = +0.835, n = 7; R = +0.983, n = 4), but the E4 correlations do not (R = -0.084, n = 11; R = +0.372, n = 10).

## ca3-sg-5-8-vs-13-17

> CA3 SG power at 5-8 months does not significantly predict CA3 SG power at 13-17 months, indicating slow individual changes.

In [ ]:
res = {}
for g in ("E3", "E4"):
    j = R5[R5.genotype == g][[SG]].join(R13[R13.genotype == g][[SG]], rsuffix="_o", how="inner")
    res[g] = A.corr_report(j[SG], j[SG + "_o"], f"{g} 5-8 vs 13-17 mo")
record("ca3-sg-5-8-vs-13-17", "reproduced" if res["E3"]["p"] >= 0.05 else "not_reproduced", measured=round(res["E3"]["r"], 3))


**Result:** Reproduced on the claim's test. In the 4 E3 mice with CA3 labels at both ages R = +0.891, p = 0.11; the E4 variant gives R = -0.275, p = 0.51 (n = 8).

## fig4g-young-ca3-slope-days-1-2

> CA3 SG power during SWRs at 5-8 months predicts slope_1_2 on the MWM at 14-18 months in apoE4-KI mice (expected R = -0.63).

In [ ]:
CL["fig4g-young-ca3-slope-days-1-2"] = corr_claim("fig4g-young-ca3-slope-days-1-2", "R5", "MWM2", "SG", "slope_1_2", -0.63, 0.063)

**Result:** Not reproduced. R = -0.032, p = 0.94, in the 8 E4 mice with CA3 power at 5-8 months and MWM at 14-18 months (paper n = 11).

## fig4h-young-ca3-latency-day-3

> CA3 SG power during SWRs at 5-8 months predicts latency_d3 on the MWM at 14-18 months in apoE4-KI mice (expected R = -0.74).

In [ ]:
CL["fig4h-young-ca3-latency-day-3"] = corr_claim("fig4h-young-ca3-latency-day-3", "R5", "MWM2", "SG", "latency_d3", -0.74, 0.074)

**Result:** Not reproduced. R = +0.074, p = 0.86 (n = 8).

## figs4h-young-ca3-slope-days-1-3

> CA3 SG power during SWRs at 5-8 months predicts slope_1_3 on the MWM at 14-18 months in apoE4-KI mice (expected R = -0.74).

In [ ]:
CL["figs4h-young-ca3-slope-days-1-3"] = corr_claim("figs4h-young-ca3-slope-days-1-3", "R5", "MWM2", "SG", "slope_1_3", -0.74, 0.074)

**Result:** Not reproduced. R = +0.180, p = 0.67 (n = 8).

## fig4jk-young-ca3-apa

> CA3 SG power during SWRs at 5-8 months predicts latency to first shock zone entrance and path length on APA day 2 at 15-20 months in apoE4-KI mice.

In [ ]:
subs = []
corr_claim("fig4jk-young-ca3-apa-0", "R5", "APA_W", "SG", "latency_first_entrance_s_d2", 0.79, 0.079); subs.append(OUT.pop("fig4jk-young-ca3-apa-0"))
corr_claim("fig4jk-young-ca3-apa-1", "R5", "APA_W", "SG", "path_length_m_d2", 0.68, 0.068); subs.append(OUT.pop("fig4jk-young-ca3-apa-1"))
o = [s_["outcome"] for s_ in subs]
out = "reproduced" if all(x == "reproduced" for x in o) else ("partially_reproduced" if any(x != "not_reproduced" for x in o) else "not_reproduced")
record("fig4jk-young-ca3-apa", out, assumed=any(s_["assumed"] for s_ in subs), measured=[s_["measured"] for s_ in subs])


**Result:** Not reproduced. Latency to first entrance R = -0.070 and path length R = -0.155 (n = 8), both with the wrong sign.

## fig4i-young-mwm-learning-score

> The MWM learning score at 14-18 months predicted from CA3 SG power at 5-8 months by the screen cohort's model accounts for 50% of the variance in the actual score.

In [ ]:
outs = {}
for sgc in SG_READINGS.values():
    outs[sgc] = pred_claim("fig4i", lambda pop: score_table(C1, MWM1, "rate_5sd", A.MWM_LEARN, A.MWM_LEARN_INV, pop),
                           lambda pop: score_table(R5, MWM2, sgc, A.MWM_LEARN, A.MWM_LEARN_INV, pop), 0.50, 0.05, sgc + " at 5-8 mo")
record("fig4i-young-mwm-learning-score", outs[SG][0], assumed=len({v[0] for v in outs.values()}) > 1, measured=outs[SG][1], readings={k: v[0] for k, v in outs.items()})


**Result:** Not reproduced. The predicted score explains 0.3% of the variance (R = -0.054, n = 8), against 50%.

## fig4l-young-apa-learning-score

> The APA learning score at 15-20 months predicted from CA3 SG power at 5-8 months by the screen cohort's model accounts for 71% of the variance in the actual score.

In [ ]:
outs = {}
mets = ["latency_first_entrance_s_d2", "path_length_m_d2", "pct_time_opp_d2"]
for sgc in SG_READINGS.values():
    outs[sgc] = pred_claim("fig4l", lambda pop: score_table(C1, MWM1, "rate_5sd", A.MWM_LEARN, A.MWM_LEARN_INV, pop),
                           lambda pop: score_table(R5, APA_W, sgc, mets, [], pop), 0.71, 0.071, sgc + " at 5-8 mo")
record("fig4l-young-apa-learning-score", outs[SG][0], assumed=len({v[0] for v in outs.values()}) > 1, measured=outs[SG][1], readings={k: v[0] for k, v in outs.items()})


**Result:** Not reproduced. The predicted score explains 0.2 to 2.0% of the variance (R between -0.14 and -0.05, n = 8), against 71%.

## table-s4-30-comparisons

> Of 30 comparisons of 15 later behavioral metrics against the two SWR properties at 5-8 months, 5 are significant at alpha 0.05, more than chance.

In [ ]:
mw = ["slope_1_2", "slope_1_3", "slope_1_4", "slope_1_5", "latency_d3", "night_1", "pct_quadrant_p1", "crossings_p1", "auc_p1"]
ap6 = ["entrances_d2", "latency_first_entrance_s_d2", "path_length_m_d2", "pct_time_opp_d2", "dpb_d2", "dpb_d3"]
print("The 15 metrics of Table S4 are taken as: MWM", mw, "and APA", ap6)
rows = []
for pc in ("rate_5sd", SG):
    for B, ms in ((MWM2, mw), (APA_W, ap6)):
        d = join(R5, B, [pc])
        for m in ms:
            r = stats.pearson(d[pc], d[m]); rows.append((pc, m, r["n"], round(r["r"], 3), r["p"]))
T30 = pd.DataFrame(rows, columns=["property", "metric", "n", "R", "p"]); k = int((T30.p < 0.05).sum())
print(T30.round(4).to_string()); print("significant:", k, "of", len(T30), "; binomial p =", round(stats.binom_greater(k, len(T30)), 4))
record("table-s4-30-comparisons", "reproduced" if abs(k - 5) <= 1 and stats.binom_greater(k, len(T30)) < 0.05 else "not_reproduced", measured=f"{k} of {len(T30)}")


**Result:** Not reproduced. 4 of 30 correlations have p < 0.05 (within the tolerance of 5 +- 1), but the binomial p is 0.061, not below 0.05; the significant ones are not the paper's (for example SWR abundance against day 2 latency to first entrance, R = -0.926, the opposite sign to the claims' direction).

## figs4i-young-ca3-predicts-later-swr

> In apoE4-KI but not apoE3-KI mice, CA3 SG power during SWRs at 5-8 months predicts SWR abundance at 13-17 months in the same mouse.

In [ ]:
res = {}
for g in ("E4", "E3"):
    j = R5[R5.genotype == g][[SG]].join(R13[R13.genotype == g][["rate_5sd"]], how="inner")
    res[g] = A.corr_report(j[SG], j.rate_5sd, f"{g}: CA3 SG 5-8 mo vs abundance 13-17 mo")
e4ok = res["E4"]["r"] > 0 and res["E4"]["p"] < 0.05 and abs(res["E4"]["r"] - 0.823) <= 0.082
out = "reproduced" if e4ok and res["E3"]["p"] >= 0.05 else ("partially_reproduced" if res["E4"]["r"] > 0 and res["E4"]["p"] < 0.05 else "not_reproduced")
record("figs4i-young-ca3-predicts-later-swr", out, measured=round(res["E4"]["r"], 3))


**Result:** Not reproduced. In E4 mice R = +0.447, p = 0.19 (n = 10; paper R = 0.823, n = 15); in E3 mice R = +0.126, p = 0.79.

## robust-single-point

> None of the paper's significant correlations is driven by a single data point; each stays significant when any one point is removed.

In [ ]:
rows = [(k, v["n"], round(v["r"], 3), round(v["p"], 4), v["loo_fail"]) for k, v in LOO.items()
        if OUT.get(k.split(" (")[0], {}).get("outcome") in ("reproduced", "partially_reproduced") and v["p"] < 0.05]
T = pd.DataFrame(rows, columns=["claim", "n", "R", "p", "removals with p > 0.05"]); print(T.to_string())
nf = T["removals with p > 0.05"]
out = "reproduced" if (nf == 0).all() else ("partially_reproduced" if (nf <= 1).all() else "not_reproduced")
record("robust-single-point", out, measured=f"{int((nf > 0).sum())} of {len(T)} correlations have a removal with p > 0.05")


**Result:** Not reproduced. Of the 7 correlations that reproduced here at least in part, 2 lose significance when one mouse is removed: figs1b (1 removal) and fig2d (5 removals). The other 5 stay below 0.05 for every removal.

## Summary

### Claims and outcomes

| Claim | Outcome |
|---|---|
| fig1e-swr-abundance-genotype | Reproduced |
| fig1f-sg-power-genotype | Reproduced |
| figs1a-mua-during-swrs | Reproduced |
| fig1g-mwm-impaired | Reproduced |
| screen-88-comparisons | Not reproduced |
| fig2b-swr-slope-days-1-2 | Not reproduced |
| figs1b-swr-slope-days-1-3 | Partially reproduced |
| fig2c-swr-latency-day-3 | Not reproduced |
| fig2d-swr-overnight | Partially reproduced |
| fig2e-learning-score | Partially reproduced |
| screen-features-independent | Not reproduced |
| fig2f-ca3-probe1-quadrant | Not reproduced |
| figs1d-ca3-probe2-quadrant | Not reproduced |
| fig2g-ca3-probe1-crossings | Partially reproduced |
| fig2h-ca3-probe1-auc | Not reproduced |
| figs1f-ca3-probe2-auc | Not reproduced |
| fig2i-memory-precision-score | Partially reproduced |
| rep-mwm-swr-slope-days-1-2 | Not reproduced |
| rep-mwm-swr-latency-day-3 | Not reproduced |
| rep-mwm-swr-overnight | Not reproduced |
| rep-mwm-swr-slope-days-1-3 | Not reproduced |
| fig2n-rep-learning-score | Not reproduced |
| rep-mwm-ca3-probe1-quadrant | Not reproduced |
| rep-mwm-ca3-probe1-crossings | Not reproduced |
| rep-mwm-ca3-probe1-auc | Not reproduced |
| rep-mwm-ca3-probe2 | Not reproduced |
| fig2r-rep-memory-score | Not reproduced |
| rep-all-replicated | Not reproduced |
| cohort-swr-differences | Reproduced |
| robust-single-point | Not reproduced |
| robust-detection-parameters | Not reproduced |
| age-no-correlation | Reproduced |
| figs2-nonspatial-no-difference | Reproduced |
| nonspatial-not-correlated | Not reproduced |
| ca1-dg-not-predictive | Not reproduced |
| e3-relationships | Not reproduced |
| fig3b-apa-entrances-day1 | Reproduced |
| figs3-apa-day1-and-probe | Partially reproduced |
| apa-40-comparisons | Not reproduced |
| apa-day2-variance | Partially reproduced |
| figs3fg-day2-predicts-later | Reproduced |
| fig3c-swr-latency-first-entrance | Not reproduced |
| fig3de-swr-path-and-quadrant | Not reproduced |
| fig3f-apa-learning-score | Not reproduced |
| fig3g-ca3-entrances | Not reproduced |
| fig3h-ca3-distance-per-bout | Not reproduced |
| fig3i-apa-memory-score | Not reproduced |
| figs3ij-mwm-predicts-apa | Partially reproduced |
| figs4ab-young-deficits | Not reproduced |
| figs4ab-aging-increase | Partially reproduced |
| figs4cd-baseline-sd-swr-band | Reproduced |
| figs4ef-ca3-sg-baseline-stable | Not reproduced |
| figs4g-young-no-deficit | Reproduced |
| table-s4-young-no-prediction | Blocked: metadata insufficient |
| fig4bc-swr-not-stable | Reproduced |
| fig4de-ca3-sg-stable | Not reproduced |
| ca3-sg-5-8-vs-13-17 | Reproduced |
| fig4g-young-ca3-slope-days-1-2 | Not reproduced |
| fig4h-young-ca3-latency-day-3 | Not reproduced |
| figs4h-young-ca3-slope-days-1-3 | Not reproduced |
| fig4i-young-mwm-learning-score | Not reproduced |
| fig4jk-young-ca3-apa | Not reproduced |
| fig4l-young-apa-learning-score | Not reproduced |
| table-s4-30-comparisons | Not reproduced |
| figs4i-young-ca3-predicts-later-swr | Not reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

### How the analysis differs from the paper's

- SWR detection follows the paper's code (filters from its repository, 4 ms smoothing, session mean and SD, 3 SD events with a 15 ms sustained level, the 16-channel noise rule, immobility of 30 s below 1 cm/s). Events are extracted by a Python port of extracteventsnew.cpp; the sustained level (maxthresh) is the best 15-sample window inside the event, where the C code searches greedily from the peak.
- SG power uses Chronux's taper settings and FFT grid, but z-scores against the immobile 100 ms windows of the session in place (the code concatenates immobile samples first).
- CA3, CA1-sr and DG sites are taken from the deposited chinfo labels. The paper's Table S1 counts (CA3 in 11 E3 and 13 E4 screen mice; 9 E3 and 14 E4 replication mice at 13-17 months) do not match the labels (29 screen mice with CA3; 4 E3 and 11 E4 at 13-17 months), and the dentate-spike check of DG sites was not run.
- The replication cohort at 5-8 months has 14 E4 mice in the data set against 17 in Table S1, so the Figure 4 correlations have 8 mice where the paper has 10 or 11.
- The 6-9 month MWM file cannot be linked to recorded mice, so `table-s4-young-no-prediction` is blocked.
- The distance-per-bout metric uses the walk bouts of pull_apa_metrics.py, the column that reproduces the printed Figure S3C statistic (U = 30); the all-bouts column is shown for fig3h.

### What was found about the data set

- The data descriptor gives MUA spike times in ms; the values are in samples at 30 kHz (as RTMUA.m assumes).
- `All_Figures.pzfx` is a binary Prism file (PCFFGRA4 header), not the XML format its extension names, so it cannot be read without Prism.
- Site labels (chinfo) change between ages for the same mouse (for example JellyBean and KitKat lose their CA3 labels at 13-17 months) and do not reproduce the CA3 membership of Table S1 in either cohort.
- Three E4 mice of Table S1 at 5-8 months and one at 9-11 months are not in the data set.
- The 6-9 month MWM spreadsheet uses its own mouse IDs, with no mapping to the recorded animals.
- The hot plate sheet lacks four E4 mice; Cohort 1 mouse 1585 (19) has no MWM data; ear tags 1386 and 1388 are shared by an E3 and an E4 mouse in Cohort 1.
- 33 sessions are marked FAIL in task.env and are excluded, as the code's epoch filter does.


In [ ]:
S_ = pd.DataFrame([{"claim": k, **{kk: v.get(kk) for kk in ("outcome", "assumed", "measured", "blocked_by")}} for k, v in OUT.items()])
print(S_.to_string())
print(S_.outcome.value_counts())
print("runtime minutes:", round((time.time() - T0) / 60, 1))
